# Train Supply XGBoost + Extra Trees Stacking Ensembles

Run the supply XGBoost and ETR notebooks first. This notebook fits a Ridge meta-model on their chronological validation predictions and evaluates it once on the untouched test period.

The stacker does not replace the two base pipelines: production inference requires the matching XGBoost pipeline, ETR pipeline, and Ridge stacker.

In [ ]:
%pip install -q pandas scikit-learn joblib matplotlib seaborn

In [ ]:
import json
import platform
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import sklearn
from google.colab import drive
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

drive.mount('/content/drive')

ROOT = Path('/content/drive/MyDrive/demand-forecasting/artifacts/supply')
VARIANTS = ('core', 'full')
TARGET_COLUMN = 'future_production_kg'
KEY_COLUMNS = [
    'forecast_origin_month', 'target_month', 'crop', 'location', 'forecast_horizon'
]
META_FEATURES = ['xgboost_prediction', 'etr_prediction']

for variant in VARIANTS:
    prediction_dir = ROOT / variant / 'predictions'
    required_files = [
        prediction_dir / 'xgboost_validation_predictions.csv',
        prediction_dir / 'xgboost_test_predictions.csv',
        prediction_dir / 'etr_validation_predictions.csv',
        prediction_dir / 'etr_test_predictions.csv',
    ]
    missing = [str(path) for path in required_files if not path.exists()]
    if missing:
        raise FileNotFoundError(
            'Run the supply XGBoost and ETR notebooks first. Missing:\n'
            + '\n'.join(missing)
        )
print('Base-model prediction files found for both variants.')

In [ ]:
def load_aligned_predictions(variant, split):
    prediction_dir = ROOT / variant / 'predictions'
    xgboost_predictions = pd.read_csv(
        prediction_dir / f'xgboost_{split}_predictions.csv',
        parse_dates=['forecast_origin_month', 'target_month'],
    )
    etr_predictions = pd.read_csv(
        prediction_dir / f'etr_{split}_predictions.csv',
        parse_dates=['forecast_origin_month', 'target_month'],
    )
    required_xgboost = KEY_COLUMNS + [TARGET_COLUMN, 'xgboost_prediction']
    required_etr = KEY_COLUMNS + [TARGET_COLUMN, 'etr_prediction']
    for name, frame, required in [
        ('XGBoost', xgboost_predictions, required_xgboost),
        ('ETR', etr_predictions, required_etr),
    ]:
        missing = [column for column in required if column not in frame.columns]
        if missing:
            raise ValueError(f'{variant} {split} {name} predictions are missing: {missing}')
        if frame.duplicated(KEY_COLUMNS).any():
            raise ValueError(f'{variant} {split} {name} predictions contain duplicate keys.')

    joined = xgboost_predictions[required_xgboost].merge(
        etr_predictions[required_etr].rename(columns={TARGET_COLUMN: 'etr_actual'}),
        on=KEY_COLUMNS,
        how='inner',
        validate='one_to_one',
    )
    if len(joined) != len(xgboost_predictions) or len(joined) != len(etr_predictions):
        raise ValueError(f'{variant} {split} base predictions do not have identical rows.')
    if not np.allclose(joined[TARGET_COLUMN], joined['etr_actual']):
        raise ValueError(f'{variant} {split} actual targets do not match.')
    if joined[META_FEATURES].isna().any().any():
        raise ValueError(f'{variant} {split} base predictions contain missing values.')
    return joined.drop(columns='etr_actual').sort_values(KEY_COLUMNS).reset_index(drop=True)


def calculate_metrics(actual, predicted):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    denominator = np.abs(actual).sum()
    return {
        'mae': float(mean_absolute_error(actual, predicted)),
        'rmse': float(np.sqrt(mean_squared_error(actual, predicted))),
        'wape': float(np.abs(actual - predicted).sum() / denominator) if denominator else np.nan,
        'r2': float(r2_score(actual, predicted)),
    }

In [ ]:
records = []
variant_results = {}

for variant in VARIANTS:
    print(f'\nTraining supply stacking ensemble for {variant}...')
    validation = load_aligned_predictions(variant, 'validation')
    test = load_aligned_predictions(variant, 'test')

    stacker = Ridge(alpha=1.0)
    stacker.fit(validation[META_FEATURES], validation[TARGET_COLUMN])
    validation['stacking_prediction'] = np.maximum(
        stacker.predict(validation[META_FEATURES]), 0
    )
    test['stacking_prediction'] = np.maximum(
        stacker.predict(test[META_FEATURES]), 0
    )

    prediction_columns = {
        'xgboost': 'xgboost_prediction',
        'extra_trees': 'etr_prediction',
        'stacking_ensemble': 'stacking_prediction',
    }
    variant_records = []
    for split_name, frame in [('validation', validation), ('test', test)]:
        for model_name, prediction_column in prediction_columns.items():
            variant_records.append({
                'model': model_name, 'model_variant': variant,
                'split': split_name, 'forecast_horizon': 'overall',
                'rows': len(frame),
                **calculate_metrics(frame[TARGET_COLUMN], frame[prediction_column]),
            })
            for horizon, group in frame.groupby('forecast_horizon'):
                variant_records.append({
                    'model': model_name, 'model_variant': variant,
                    'split': split_name, 'forecast_horizon': int(horizon),
                    'rows': len(group),
                    **calculate_metrics(group[TARGET_COLUMN], group[prediction_column]),
                })
    records.extend(variant_records)

    model_dir = ROOT / variant / 'models'
    prediction_dir = ROOT / variant / 'predictions'
    report_dir = ROOT / variant / 'reports'
    for directory in (model_dir, prediction_dir, report_dir):
        directory.mkdir(parents=True, exist_ok=True)

    joblib.dump(stacker, model_dir / 'experimental_ridge_stacker.joblib')
    validation.to_csv(
        prediction_dir / 'stacking_validation_comparison.csv', index=False
    )
    test.to_csv(prediction_dir / 'stacking_test_comparison.csv', index=False)
    pd.DataFrame(variant_records).to_csv(
        report_dir / 'stacking_model_comparison_metrics.csv', index=False
    )

    test_stacking_metrics = next(
        row for row in variant_records
        if row['model'] == 'stacking_ensemble'
        and row['split'] == 'test'
        and row['forecast_horizon'] == 'overall'
    )
    metadata = {
        'model': 'experimental_supply_ridge_stacker',
        'model_variant': variant,
        'status': 'experimental',
        'requires_air_quality': variant == 'full',
        'synthetic_training_target': True,
        'target': TARGET_COLUMN,
        'base_models': ['global_xgboost_pipeline.joblib', 'global_etr_pipeline.joblib'],
        'meta_features': META_FEATURES,
        'ridge_alpha': float(stacker.alpha),
        'ridge_intercept': float(stacker.intercept_),
        'ridge_coefficients': dict(zip(META_FEATURES, map(float, stacker.coef_))),
        'test_metrics': test_stacking_metrics,
        'versions': {
            'python': platform.python_version(),
            'pandas': pd.__version__,
            'scikit_learn': sklearn.__version__,
            'joblib': joblib.__version__,
        },
    }
    (report_dir / 'stacking_training_metadata.json').write_text(
        json.dumps(metadata, indent=2), encoding='utf-8'
    )
    variant_results[variant] = test
    print(f'Saved {variant} stacking artifacts to: {ROOT / variant}')

metrics = pd.DataFrame(records)
shared_report_dir = ROOT / 'reports'
shared_report_dir.mkdir(parents=True, exist_ok=True)
metrics.to_csv(shared_report_dir / 'supply_full_core_model_comparison.csv', index=False)
print('\nSUPPLY STACKING TRAINING COMPLETE')

In [ ]:
overall_test = metrics[
    (metrics['split'] == 'test')
    & (metrics['forecast_horizon'] == 'overall')
].copy()
overall_test['wape_percent'] = overall_test['wape'] * 100
display(
    overall_test[
        ['model_variant', 'model', 'mae', 'rmse', 'wape_percent', 'r2']
    ].sort_values(['model_variant', 'mae']).style.format({
        'mae': '{:,.2f}', 'rmse': '{:,.2f}',
        'wape_percent': '{:.2f}%', 'r2': '{:.3f}',
    })
)

sns.set_theme(style='whitegrid')
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
sns.barplot(
    data=overall_test, x='model', y='mae', hue='model_variant', ax=axes[0]
)
axes[0].set_title('Supply Model Test MAE')
axes[0].set_xlabel('Model')
axes[0].set_ylabel('MAE (kg)')
axes[0].tick_params(axis='x', rotation=15)

horizon_metrics = metrics[
    (metrics['split'] == 'test')
    & (metrics['forecast_horizon'] != 'overall')
].copy()
horizon_metrics['forecast_horizon'] = horizon_metrics['forecast_horizon'].astype(int)
sns.lineplot(
    data=horizon_metrics, x='forecast_horizon', y='mae',
    hue='model', style='model_variant', marker='o', ax=axes[1],
)
axes[1].set_title('Supply Test MAE by Horizon and Variant')
axes[1].set_xlabel('Forecast horizon')
axes[1].set_ylabel('MAE (kg)')
axes[1].set_xticks(range(1, 13))
plt.tight_layout()
comparison_figure = shared_report_dir / 'supply_model_comparison.png'
plt.savefig(comparison_figure, dpi=200, bbox_inches='tight')
plt.show()

plt.figure(figsize=(9, 5))
sns.barplot(
    data=overall_test, x='model', y='r2', hue='model_variant'
)
plt.title('Supply Model Test R²')
plt.xlabel('Model')
plt.ylabel('R²')
plt.xticks(rotation=15)
plt.tight_layout()
r2_figure = shared_report_dir / 'supply_model_r2_comparison.png'
plt.savefig(r2_figure, dpi=200, bbox_inches='tight')
plt.show()

print(f'Comparison table: {shared_report_dir / "supply_full_core_model_comparison.csv"}')
print(f'Comparison figures: {comparison_figure} and {r2_figure}')

## Selecting the supply model

Choose the production candidate from the held-out test results, with lower MAE/RMSE/WAPE and higher R² preferred. Do not assume stacking is automatically best. Because this experiment uses synthetic production targets, the selected model remains a prototype until retrained and re-evaluated with real local production records.